In [17]:
!pip -q install streamlit pandas scikit-learn numpy


In [18]:
!wget -q -O ngrok.tgz https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz
!tar -xzf ngrok.tgz
!chmod +x ngrok
!./ngrok --version


ngrok version 3.30.0


In [19]:
NGROK_AUTH_TOKEN = "329c1zZsIBpyO0DbUFli965tZte_4Kk6opJzr7Kx5w85rUKPx"  # <-- replace this
import subprocess, shlex, os, sys

assert NGROK_AUTH_TOKEN and NGROK_AUTH_TOKEN != "PASTE_YOUR_NGROK_TOKEN_HERE", "Put your real ngrok token first."
# Store token in ngrok's config
subprocess.run(["./ngrok", "config", "add-authtoken", NGROK_AUTH_TOKEN], check=True)
print("✅ ngrok token saved")


✅ ngrok token saved


In [20]:
import os
print("Files:", os.listdir())
assert os.path.exists("app.py"), "❌ app.py not found in this folder."
print("✅ app.py found")


Files: ['.config', 'Intern_Skills.csv', 'Skills_Ontology.csv', 'drive', '.ipynb_checkpoints', 'ngrok', 'ngrok.tgz', 'Job_Descriptions.csv', 'app.py', 'sample_data']
✅ app.py found


In [23]:
import subprocess, time, re, sys

# Kill any previous runs
!pkill -f "streamlit run app.py" || true
!pkill -f ngrok || true

# Start Streamlit headless on 8501
sp_app = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

time.sleep(4)  # give Streamlit some time to boot

# Start ngrok v3; set region near PK (try "in" or "ap")
sp_tun = subprocess.Popen(
    ["./ngrok", "http", "8501", "--region", "in", "--log", "stdout", "--log-format", "logfmt"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

print("⏳ Waiting for public URL...")
public_url = None
while True:
    line = sp_tun.stdout.readline()
    if not line:
        break
    line = line.strip()
    # print logs to help debugging
    print(line)
    # ngrok v3 logfmt includes 'url=https://...'
    m = re.search(r"url=(https://[^\\s]+)", line)
    if m:
        public_url = m.group(1)
        # Prefer the https tunnel (ignore http one if it appears first)
        if public_url.startswith("https://"):
            print("\n✅ Public URL:", public_url)
            break

# Optionally show initial Streamlit logs
print("\n--- Streamlit logs (first lines) ---")
for _ in range(20):
    l = sp_app.stdout.readline()
    if not l: break
    print(l.rstrip())


^C
^C
⏳ Waiting for public URL...
t=2025-10-07T05:58:38+0000 lvl=info msg="command usage" msg="Flag --region has been deprecated, ngrok automatically chooses the region with lowest latency"
t=2025-10-07T05:58:38+0000 lvl=info msg="no configuration paths supplied"
t=2025-10-07T05:58:38+0000 lvl=warn msg="ngrok config file found at legacy location, move to XDG location" xdg_path=/root/.config/ngrok/ngrok.yml legacy_path=/root/.ngrok2/ngrok.yml
t=2025-10-07T05:58:38+0000 lvl=info msg="using configuration at default config path" path=/root/.ngrok2/ngrok.yml
t=2025-10-07T05:58:38+0000 lvl=info msg="open config file" path=/root/.ngrok2/ngrok.yml err=nil
t=2025-10-07T05:58:38+0000 lvl=info msg="starting web service" obj=web addr=127.0.0.1:4040 allow_hosts=[]
t=2025-10-07T05:58:39+0000 lvl=info msg="client session established" obj=tunnels.session
t=2025-10-07T05:58:39+0000 lvl=info msg="tunnel session started" obj=tunnels.session
t=2025-10-07T05:58:39+0000 lvl=info msg="started tunnel" obj=tun

In [22]:
# Kill common processes (ignore errors if not running)
!pkill -f "streamlit run app.py" || true
!pkill -f ngrok || true
!pkill -f cloudflared || true

# Extra safety: free port 8501 (Linux)
import subprocess, re, os, signal, sys

def kill_by_port(port=8501):
    try:
        out = subprocess.check_output(["bash","-lc", f"lsof -i:{port} -t"], text=True).strip()
        if out:
            for pid in set(re.findall(r"\d+", out)):
                try:
                    os.kill(int(pid), signal.SIGKILL)
                    print(f"Killed PID {pid} on port {port}")
                except Exception as e:
                    print(f"Could not kill PID {pid}: {e}")
        else:
            print(f"No process found on :{port}")
    except subprocess.CalledProcessError:
        print(f"No process found on :{port}")

kill_by_port(8501)

print("✅ Cleaned up Streamlit/tunnels. You can relaunch now.")


^C
^C
^C
No process found on :8501
✅ Cleaned up Streamlit/tunnels. You can relaunch now.
